# GRN/regulon baseline

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
    "#000000",
    "#ffcc00",
    "#8b4513",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
    "schwarz",
    "goldgelb",
    "sattelbraun",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "fig6_neurips2021_cite_bmmc_grn")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import textwrap
from adjustText import adjust_text
import matplotlib.patches as mpatches
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


In [ ]:
_autosave_counter = 0

def show_umap_plot(X_umap, labels, cmap, encode_first_as_grey=False, continuous=False, title='', figsize=(3, 3), save_name=False):
    set_figure_params(4, 4)
    if not continuous:
        unique_labels, encoded_labels = np.unique(labels, return_inverse=True)
        n_categories = len(unique_labels)
        base_palette = sns.color_palette('husl', n_categories)
        if n_categories > 10:
            reordered_palette = []
            spacing = 6
            for i in range(len(base_palette)):
                index = i // spacing + i % spacing * (len(base_palette) // spacing)
                reordered_palette.append(base_palette[index % n_categories])
        else:
            reordered_palette = base_palette
        if not encode_first_as_grey:
            new_palette = reordered_palette
            cmap = ListedColormap(new_palette)
        else:
            new_palette = ['#eff1f2'] + reordered_palette[1:]
            cmap = ListedColormap(new_palette)
    else:
        encoded_labels = labels
    fig, ax = plt.subplots()
    ax.set_aspect('equal', 'box')
    if continuous:
        sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, cmap=cmap, c=encoded_labels, edgecolor='none')
    else:
        sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, c=encoded_labels, cmap=cmap, edgecolor='none')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_box_aspect(1)
    plt.title(title)
    global _autosave_counter
    if not save_name:
        _autosave_counter += 1
        save_name = f'autosave_{_autosave_counter:03d}'
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.pdf'), dpi=500, transparent=True, bbox_inches='tight', format='pdf')
    plt.show()
    plt.close()

def show_umap_plot(X_umap, labels, cmap, encode_first_as_grey=False, continuous=False, title='', figsize=(3, 3), save_name=False):
    set_figure_params(4, 4)
    unique_labels = np.unique(labels)
    label_to_int = {label: i for i, label in enumerate(unique_labels)}
    encoded_labels = [label_to_int[l] for l in labels]
    colors = [type_to_color.get(label, '#9E9E9E') for label in unique_labels]
    cmap = ListedColormap(colors)
    fig, ax = plt.subplots()
    ax.set_aspect('equal', 'box')
    sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, c=encoded_labels, cmap=cmap, edgecolor='none')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_box_aspect(1)
    plt.title(title)
    global _autosave_counter
    if not save_name:
        _autosave_counter += 1
        save_name = f'autosave_{_autosave_counter:03d}'
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.pdf'), dpi=500, transparent=True, bbox_inches='tight', format='pdf')
    plt.show()
    plt.close()

def show_colorbar(cmap, save_name):
    fig, ax = plt.subplots(figsize=(1.5, 4))
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1))
    cbar = plt.colorbar(sm, ax=ax)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label('GO activation', rotation=270, labelpad=15)
    plt.axis('off')
    global _autosave_counter
    if not save_name:
        _autosave_counter += 1
        save_name = f'autosave_{_autosave_counter:03d}'
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.pdf'), dpi=500, transparent=True, bbox_inches='tight', format='pdf')
    plt.show()

def show_legend(labels, save_name):
    unique_labels, encoded_labels = np.unique(labels, return_inverse=True)
    n_categories = len(unique_labels)
    base_palette = sns.color_palette('husl', n_categories)
    if n_categories > 10:
        reordered_palette = []
        spacing = 6
        for i in range(len(base_palette)):
            index = i // spacing + i % spacing * (len(base_palette) // spacing)
            reordered_palette.append(base_palette[index % n_categories])
    else:
        reordered_palette = base_palette
    new_palette = reordered_palette
    cmap = ListedColormap(new_palette)
    fig, ax = plt.subplots(figsize=(2, 8))
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=cmap(i)) for i in range(n_categories)]
    ax.legend(handles, unique_labels, bbox_to_anchor=(1, 1), loc='upper left')
    plt.axis('off')
    global _autosave_counter
    if not save_name:
        _autosave_counter += 1
        save_name = f'autosave_{_autosave_counter:03d}'
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.pdf'), dpi=500, transparent=True, bbox_inches='tight', format='pdf')
    plt.show()

def show_legend(labels, save_name=False):
    unique_labels = sorted(set(labels))
    colors = [type_to_color.get(label, '#9E9E9E') for label in unique_labels]
    fig, ax = plt.subplots(figsize=(2, min(0.3 * len(unique_labels), 15)))
    handles = [plt.Line2D([0], [0], marker='o', linestyle='None', markersize=6, markerfacecolor=color, markeredgecolor='k') for color in colors]
    ax.legend(handles, unique_labels, bbox_to_anchor=(1, 1), loc='upper left', frameon=False)
    plt.axis('off')
    global _autosave_counter
    if not save_name:
        _autosave_counter += 1
        save_name = f'autosave_{_autosave_counter:03d}'
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.pdf'), dpi=500, transparent=True, bbox_inches='tight', format='pdf')
    plt.show()
    plt.close()

def show_legend(labels, save_name=False):
    family_to_labels = defaultdict(list)
    for label in np.unique(labels):
        family = map_cell_type_to_family(label)
        family_to_labels[family].append(label)
    family_order = list(family_colors.keys())
    handles = []
    legend_labels = []
    for family in family_order:
        for label in sorted(family_to_labels.get(family, [])):
            color = type_to_color.get(label, '#9E9E9E')
            handles.append(plt.Line2D([0], [0], marker='o', linestyle='None', markersize=6, markerfacecolor=color, markeredgecolor='k'))
            legend_labels.append(f'{label}')
    fig, ax = plt.subplots()
    ax.legend(handles, legend_labels, bbox_to_anchor=(1, 1), loc='upper left', frameon=False)
    plt.axis('off')
    global _autosave_counter
    if not save_name:
        _autosave_counter += 1
        save_name = f'autosave_{_autosave_counter:03d}'
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.pdf'), dpi=500, transparent=True, bbox_inches='tight', format='pdf')
    plt.show()
    plt.close()


In [ ]:
(
    mdata,
    mdata_counts,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_cite_BMMC(1.0, None, True)


In [ ]:
def map_cell_type_to_family(cell_type):
    cell_type = cell_type.lower()

    if any(x in cell_type for x in ["hsc", "prog"]):
        return "Progenitor"
    elif "b" in cell_type:
        return "B cell"
    elif any(x in cell_type for x in ["nk", "ilc"]):
        return "ILC/NK"
    elif any(x in cell_type for x in ["cd4", "cd8", "treg", "dnT", "gdT", "mait"]):
        return "T cell"
    elif "mono" in cell_type or "mac" in cell_type:
        return "Monocyte"
    elif "dc" in cell_type:
        return "Dendritic cell"
    elif any(x in cell_type for x in ["erythro", "nomo", "retic", "normo"]):
        return "Erythroid"
    elif any(x in cell_type for x in ["plasmablast", "plasma"]):
        return "B cell"
    elif "platelet" in cell_type:
        return "Platelet"
    elif "doublet" in cell_type:
        return "Artifact"
    else:
        return "Other"

family_colors = {
    "Progenitor": "#4CAF50",
    "B cell": "#E53935",
    "ILC/NK": "#FB8C00",
    "T cell": "#FDD835",
    "Monocyte": "#F06292",
    "Dendritic cell": "#8E24AA",
    "Erythroid": "#6D4C41",
    "Platelet": "#90A4AE",
    "Artifact": "#BDBDBD",
    "Other": "#9E9E9E",
}

from matplotlib.colors import to_rgb, to_hex
import colorsys

def generate_family_shades(family_labels, base_color):
    '''Generate different shades for subtypes in a given family.'''
    base_rgb = to_rgb(base_color)
    hsv = colorsys.rgb_to_hsv(*base_rgb)
    n = len(family_labels)
    shades = []

    for i in range(n):
        value = 0.6 + (0.4 * (i / max(n - 1, 1)))
        rgb = colorsys.hsv_to_rgb(hsv[0], hsv[1], value)
        shades.append(to_hex(rgb))

    return dict(zip(family_labels, shades))

def generate_family_shades(family_labels, base_color, hue_shift=0.08):
    '''Generate visually distinct colors within a family by shifting hue slightly.'''
    base_rgb = to_rgb(base_color)
    h, s, v = colorsys.rgb_to_hsv(*base_rgb)
    n = len(family_labels)
    shades = []

    for i in range(n):
        new_h = (h + hue_shift * (i / max(n - 1, 1)) - hue_shift / 2) % 1.0
        new_s = min(max(s * (0.8 + 0.2 * (i / max(n - 1, 1))), 0), 1)
        new_v = min(max(v * (0.8 + 0.2 * (i / max(n - 1, 1))), 0), 1)
        rgb = colorsys.hsv_to_rgb(new_h, new_s, new_v)
        shades.append(to_hex(rgb))

    return dict(zip(family_labels, shades))

from collections import defaultdict

def build_color_map(cell_types):
    family_to_types = defaultdict(list)
    for ct in cell_types:
        family = map_cell_type_to_family(ct)
        family_to_types[family].append(ct)

    type_to_color = {}
    for family, types in family_to_types.items():
        base_color = family_colors.get(family, "#9E9E9E")
        shades = generate_family_shades(types, base_color)
        type_to_color.update(shades)

    return type_to_color

all_cell_types = list(np.unique(mdata.mod["rna"].obs[labels_key]))
type_to_color = build_color_map(all_cell_types)


In [ ]:
pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/grn_edges.csv"
)


In [ ]:
pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/comparison/summary_table.csv"
)


In [ ]:
pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/comparison/go_overlap_networkvi.csv"
)


In [ ]:
grnboost2_adjacencies = pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/grnboost2/grnboost2_adjacencies.csv"
)
grnboost2_adjacencies


In [ ]:
grnboost2_edges = pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/grnboost2/grnboost2_edges.csv"
)
grnboost2_edges


In [ ]:
grn_adjacencies = pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/per_celltype/CD14+ Mono/method_b/grn_adjacencies.csv"
)
grn_adjacencies


In [ ]:
pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/per_celltype/CD14+ Mono/method_b/grn_hub_tfs.csv"
)


In [ ]:
pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/per_celltype/CD14+ Mono/method_c/expression_matrix.csv"
)


In [ ]:
pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/per_celltype/CD14+ Mono/method_c/scenic_adjacencies.tsv",
    delimiter="\t",
)


In [ ]:
pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/per_celltype/CD14+ Mono/method_c/scenic_auc_matrix.csv"
)


In [ ]:
pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/per_celltype/CD14+ Mono/method_c/scenic_regulons.csv"
)


In [ ]:
pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/per_celltype/CD14+ Mono/method_c/scenic_regulons.parsed.csv"
)


In [ ]:
pd.read_csv(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/per_celltype/CD14+ Mono/method_c/scenic_substate_hdbscan.csv"
)


In [ ]:
import pandas as pd
from pathlib import Path

OUTDIR = Path("standard_grn_output")
OUTDIR = Path(
    "../.experiment_data/primary/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_False_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/"
)

CELL_TYPES = [
    "CD14+ Mono",
    "CD16+ Mono",
    "NK",
    "pDC",
    "Naive CD20+ B IGKC+",
]

def safe_read(path, **kwargs):
    if path.exists():
        return pd.read_csv(path, **kwargs)
    else:
        print(f"[MISSING] {path}")
        return None

def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

print_header("1. SUMMARY TABLE (MAIN RESULT)")

summary_path = OUTDIR / "comparison" / "summary_table.csv"
summary_df = safe_read(summary_path)

if summary_df is not None:
    display(summary_df)

    print("\nINTERPRETATION GUIDE:")
    print("- found_by_any_standard = low -> NetworkVI unique biology")
    print("- leiden_max_ari_vs_substates ~0 -> clustering fails")
    print("- scenic_auc_max_ari_vs_substates low -> SCENIC fails")

print_header("2. GO TERM RECOVERY")

go_path = OUTDIR / "comparison" / "go_overlap_networkvi.csv"
go_df = safe_read(go_path)

if go_df is not None:
    display(go_df)

    print("\nKEY CHECK:")
    print("Look at GO:0002457 (antigen presentation)")
    print("If method_* columns are False -> ONLY NetworkVI finds it")

for ct in CELL_TYPES:
    print_header(f"CELL TYPE: {ct}")

    ct_key = ct.replace(" ", "_").replace("+", "pos")
    base = OUTDIR / "per_celltype" / ct

    print("\n[Method A] Marker enrichment")

    ora = safe_read(base / "method_a" / "go_enrichment_markers_ora.csv")
    if ora is not None:
        print("Top ORA terms:")
        display(ora.head(5))

    gsea = safe_read(base / "method_a" / "go_gsea_markers.csv")
    if gsea is not None:
        print("Top GSEA terms:")
        display(gsea.head(5))

    print("\n[Method B] GRNBoost2")

    hub = safe_read(base / "method_b" / "grn_hub_tfs.csv")
    if hub is not None:
        print("Top hub TFs:")
        display(hub.head(10))

    agg = safe_read(base / "method_b" / "go_enrichment_targets_aggregated.csv", index_col=0)
    if agg is not None:
        print("Top GO terms from GRN targets:")
        display(agg.head(10))

    print("INTERPRETATION:")
    print("- If GO terms here are generic -> GRN is not capturing specific biology")

    print("\n[Method C] SCENIC")

    scenic_go = safe_read(base / "method_c" / "scenic_regulon_go_enrichment.csv")
    if scenic_go is not None:
        print("Top regulon GO terms:")
        display(scenic_go.head(10))

    auc = safe_read(base / "method_c" / "scenic_auc_matrix.csv", index_col=0)
    if auc is not None:
        print(f"AUC matrix shape: {auc.shape}")

    substate = safe_read(base / "method_c" / "scenic_substate_hdbscan.csv")
    if substate is not None:
        display(substate)

        print("INTERPRETATION:")
        print("- n_hdbscan_clusters > 1 -> SCENIC finds structure")
        print("- ari_vs_pseudolabel low -> does NOT match your substates")
        print("- silhouette low -> weak separation")

    print("\n[Clustering baseline] Leiden vs substates")

    leiden = safe_read(base / "leiden_substate_ari.csv")
    if leiden is not None:
        display(leiden)

        print("INTERPRETATION:")
        print("- ARI ~0 -> clustering completely fails to recover substates")

print_header("4. AUTOMATED CLAIMS")

if summary_df is not None:
    for _, row in summary_df.iterrows():
        ct = row["cell_type"]

        print(f"\n{ct}")

        if row["found_by_any_standard"] == 0:
            print("-> NetworkVI UNIQUE: standard methods fail to recover GO biology")

        if row["leiden_max_ari_vs_substates"] == 0 or row["leiden_max_ari_vs_substates"] == "N/A":
            print("-> Clustering fails to detect substates")

        if (
            row["scenic_auc_max_ari_vs_substates"] == 0
            or row["scenic_auc_max_ari_vs_substates"] == "N/A"
        ):
            print("-> SCENIC fails to align with substates")

        print("-> NetworkVI provides added value here")

print("\nDONE. You now know exactly where to look and what to claim.")
